# 04. 모델링 (Modeling)

**모델 스택**: Logistic Regression (베이스라인) → Decision Tree → Random Forest → LightGBM → XGBoost  
**검증 방법**: Stratified K-Fold (k=5), SMOTE는 각 fold 내부에서 적용  
**주 평가 지표**: AUC-ROC  
**부 평가 지표**: F1-score, Precision, Recall

---
## 0. 라이브러리 & 설정

In [63]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, roc_curve
)
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_theme(font_scale=1.5)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
os.makedirs('../outputs', exist_ok=True)
print('라이브러리 로드 완료')

라이브러리 로드 완료


---
## 1. 데이터 로드

In [64]:
X_train = pd.read_parquet('../data/interim/X_train.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')['label']
X_test  = pd.read_parquet('../data/interim/X_test.parquet')
y_test  = pd.read_parquet('../data/interim/y_test.parquet')['label']

feature_cols = X_train.columns.tolist()

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'피처: {feature_cols}')
print(f'\nTrain 클래스 분포: {y_train.value_counts().to_dict()}')
print(f'Test  클래스 분포: {y_test.value_counts().to_dict()}')

X_train: (1562, 16)  X_test: (391, 16)
피처: ['avg_delay', 'delay_order_count', 'total_orders', 'recency_days', 'delay_trend', 'order_interval_days', 'pct_low_review', 'has_reviewed', 'total_spend', 'avg_installments', 'uses_credit_card', 'n_unique_categories', 'n_unique_sellers', 'avg_freight_ratio', 'avg_items_per_order', 'state_code']

Train 클래스 분포: {1: 1516, 0: 46}
Test  클래스 분포: {1: 379, 0: 12}


---
## 2. CV 헬퍼 & SKF 설정

- SMOTE는 각 fold의 train split에만 적용 (pipeline 내부에서 처리)
- `predict_proba`가 없는 모델 대비 `decision_function` fallback 불필요 — 세 모델 모두 지원

In [65]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(pipeline, X, y, model_name):
    """Stratified K-Fold CV. 불균형 처리는 파이프라인 내 모델 파라미터에서 처리."""
    results = {'auc': [], 'f1': [], 'precision': [], 'recall': []}
    print(f'=== {model_name} — CV (k=5) ===')
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        pipeline.fit(X_tr, y_tr)
        y_prob = pipeline.predict_proba(X_val)[:, 1]
        y_pred = pipeline.predict(X_val)
        results['auc'].append(roc_auc_score(y_val, y_prob))
        results['f1'].append(f1_score(y_val, y_pred, zero_division=0))
        results['precision'].append(precision_score(y_val, y_pred, zero_division=0))
        results['recall'].append(recall_score(y_val, y_pred, zero_division=0))
        print(f'  Fold {fold}: AUC={results["auc"][-1]:.4f}  F1={results["f1"][-1]:.4f}')
    print(f'  → CV AUC: {np.mean(results["auc"]):.4f} ± {np.std(results["auc"]):.4f}\n')
    return results


def eval_test(pipeline, X_test, y_test):
    """Test set 최종 평가."""
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = pipeline.predict(X_test)
    return {
        'test_auc'      : roc_auc_score(y_test, y_prob),
        'test_f1'       : f1_score(y_test, y_pred, zero_division=0),
        'test_precision': precision_score(y_test, y_pred, zero_division=0),
        'test_recall'   : recall_score(y_test, y_pred, zero_division=0),
        'y_prob'        : y_prob,
    }

print('헬퍼 함수 정의 완료')

헬퍼 함수 정의 완료


---
## 3. Logistic Regression

- Pipeline: SMOTE → StandardScaler → LogisticRegression
- SMOTE로 불균형 처리

In [66]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, random_state=42,
                                   class_weight='balanced')),
])

lr_cv = run_cv(lr_pipe, X_train, y_train, 'Logistic Regression')

=== Logistic Regression — CV (k=5) ===
  Fold 1: AUC=0.5753  F1=0.8340
  Fold 2: AUC=0.6416  F1=0.8311
  Fold 3: AUC=0.5787  F1=0.8138
  Fold 4: AUC=0.6480  F1=0.8399
  Fold 5: AUC=0.7125  F1=0.8512
  → CV AUC: 0.6312 ± 0.0508



In [67]:
# 전체 train으로 최종 학습 후 test 평가
lr_pipe.fit(X_train, y_train)
lr_test = eval_test(lr_pipe, X_test, y_test)

print('=== Logistic Regression — Test 결과 ===')
for k, v in lr_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== Logistic Regression — Test 결과 ===
  test_auc: 0.8210
  test_f1: 0.8215
  test_precision: 0.9852
  test_recall: 0.7045


---
## 4. LightGBM

In [68]:
lgb_pipe = Pipeline([
    ('lgb', lgb.LGBMClassifier(n_estimators=300, random_state=42,
                                 n_jobs=-1, verbose=-1,
                                 class_weight='balanced')),
])

lgb_cv = run_cv(lgb_pipe, X_train, y_train, 'LightGBM')

=== LightGBM — CV (k=5) ===
  Fold 1: AUC=0.7332  F1=0.9854
  Fold 2: AUC=0.6690  F1=0.9821
  Fold 3: AUC=0.5537  F1=0.9854
  Fold 4: AUC=0.5504  F1=0.9837
  Fold 5: AUC=0.6527  F1=0.9837
  → CV AUC: 0.6318 ± 0.0705



In [69]:
lgb_pipe.fit(X_train, y_train)
lgb_test = eval_test(lgb_pipe, X_test, y_test)

print('=== LightGBM — Test 결과 ===')
for k, v in lgb_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== LightGBM — Test 결과 ===
  test_auc: 0.7898
  test_f1: 0.9804
  test_precision: 0.9691
  test_recall: 0.9921


---
## 5. XGBoost

In [70]:
# scale_pos_weight: label=1(이탈)이 다수 → 소수 클래스(유지, label=0) 균형 보정
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train.values)
spw = float(cw[1])  # label=1의 가중치 (다수 클래스 → < 1)
print(f'scale_pos_weight: {spw:.4f}  (유지 가중치: {cw[0]:.4f})')

xgb_pipe = Pipeline([
    ('xgb', xgb.XGBClassifier(n_estimators=300, random_state=42,
                                 n_jobs=-1, verbosity=0, eval_metric='auc',
                                 scale_pos_weight=spw)),
])

xgb_cv = run_cv(xgb_pipe, X_train, y_train, 'XGBoost')

scale_pos_weight: 0.5152  (유지 가중치: 16.9783)
=== XGBoost — CV (k=5) ===
  Fold 1: AUC=0.7683  F1=0.9838
  Fold 2: AUC=0.5825  F1=0.9821
  Fold 3: AUC=0.5669  F1=0.9837
  Fold 4: AUC=0.5083  F1=0.9821
  Fold 5: AUC=0.6316  F1=0.9854
  → CV AUC: 0.6115 ± 0.0877



In [71]:
xgb_pipe.fit(X_train, y_train)
xgb_test = eval_test(xgb_pipe, X_test, y_test)

print('=== XGBoost — Test 결과 ===')
for k, v in xgb_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== XGBoost — Test 결과 ===
  test_auc: 0.7874
  test_f1: 0.9818
  test_precision: 0.9692
  test_recall: 0.9947


---
## 6. Decision Tree

In [72]:
dt_pipe = Pipeline([
    ('dt', DecisionTreeClassifier(random_state=42, class_weight='balanced')),
])

dt_cv = run_cv(dt_pipe, X_train, y_train, 'Decision Tree')

=== Decision Tree — CV (k=5) ===
  Fold 1: AUC=0.4836  F1=0.9687
  Fold 2: AUC=0.4868  F1=0.9704
  Fold 3: AUC=0.5946  F1=0.9718
  Fold 4: AUC=0.4851  F1=0.9703
  Fold 5: AUC=0.4851  F1=0.9703
  → CV AUC: 0.5071 ± 0.0438



In [73]:
dt_pipe.fit(X_train, y_train)
dt_test = eval_test(dt_pipe, X_test, y_test)

print('=== Decision Tree — Test 결과 ===')
for k, v in dt_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== Decision Tree — Test 결과 ===
  test_auc: 0.4842
  test_f1: 0.9683
  test_precision: 0.9683
  test_recall: 0.9683


---
## 7. Random Forest

In [74]:
rf_pipe = Pipeline([
    ('rf', RandomForestClassifier(n_estimators=300, random_state=42,
                                    n_jobs=-1, class_weight='balanced')),
])

rf_cv = run_cv(rf_pipe, X_train, y_train, 'Random Forest')

=== Random Forest — CV (k=5) ===
  Fold 1: AUC=0.6972  F1=0.9854
  Fold 2: AUC=0.7292  F1=0.9838
  Fold 3: AUC=0.6109  F1=0.9854
  Fold 4: AUC=0.6448  F1=0.9854
  Fold 5: AUC=0.6755  F1=0.9854
  → CV AUC: 0.6715 ± 0.0410



In [75]:
rf_pipe.fit(X_train, y_train)
rf_test = eval_test(rf_pipe, X_test, y_test)

print('=== Random Forest — Test 결과 ===')
for k, v in rf_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== Random Forest — Test 결과 ===
  test_auc: 0.8254
  test_f1: 0.9844
  test_precision: 0.9693
  test_recall: 1.0000


---
## 8. 모델 비교

In [76]:
models = {
    'Logistic Regression': (lr_cv, lr_test),
    'Decision Tree'      : (dt_cv, dt_test),
    'Random Forest'      : (rf_cv, rf_test),
    'LightGBM'           : (lgb_cv, lgb_test),
    'XGBoost'            : (xgb_cv, xgb_test),
}

rows = []
for name, (cv, test) in models.items():
    rows.append({
        '모델'          : name,
        'CV AUC (mean)' : f"{np.mean(cv['auc']):.4f}",
        'CV AUC (std)'  : f"{np.std(cv['auc']):.4f}",
        'Test AUC'      : f"{test['test_auc']:.4f}",
        'Test F1'       : f"{test['test_f1']:.4f}",
        'Test Precision': f"{test['test_precision']:.4f}",
        'Test Recall'   : f"{test['test_recall']:.4f}",
    })

comp_df = pd.DataFrame(rows).set_index('모델')
print('=== 모델 비교 ===')
print(comp_df.to_string())

=== 모델 비교 ===
                    CV AUC (mean) CV AUC (std) Test AUC Test F1 Test Precision Test Recall
모델                                                                                        
Logistic Regression        0.6312       0.0508   0.8210  0.8215         0.9852      0.7045
Decision Tree              0.5071       0.0438   0.4842  0.9683         0.9683      0.9683
Random Forest              0.6715       0.0410   0.8254  0.9844         0.9693      1.0000
LightGBM                   0.6318       0.0705   0.7898  0.9804         0.9691      0.9921
XGBoost                    0.6115       0.0877   0.7874  0.9818         0.9692      0.9947


---
## 9. Soft Voting 앙상블

- 구성: Logistic Regression + LightGBM + XGBoost
- 방식: 각 모델의 `predict_proba` 평균 (soft voting)
- 근거: LR은 선형 경계, LGB/XGB는 비선형 트리 — 서로 다른 오류를 보완

In [77]:
from sklearn.ensemble import VotingClassifier

ensemble_pipe = VotingClassifier(
    estimators=[
        ('lr',  lr_pipe),
        ('lgb', lgb_pipe),
        ('xgb', xgb_pipe),
    ],
    voting='soft',
)

ensemble_cv = run_cv(ensemble_pipe, X_train, y_train, 'Soft Voting Ensemble')
ensemble_pipe.fit(X_train, y_train)
ensemble_test = eval_test(ensemble_pipe, X_test, y_test)

print('=== Soft Voting Ensemble — Test 결과 ===')
for k, v in ensemble_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

print(f'\n[비교] Best 단일 모델 (LR) Test AUC : {lr_test["test_auc"]:.4f}')
print(f'[비교] Soft Voting Ensemble Test AUC: {ensemble_test["test_auc"]:.4f}')

=== Soft Voting Ensemble — CV (k=5) ===
  Fold 1: AUC=0.5830  F1=0.9854
  Fold 2: AUC=0.6310  F1=0.9838
  Fold 3: AUC=0.5790  F1=0.9854
  Fold 4: AUC=0.6373  F1=0.9837
  Fold 5: AUC=0.7037  F1=0.9837
  → CV AUC: 0.6268 ± 0.0453

=== Soft Voting Ensemble — Test 결과 ===
  test_auc: 0.8283
  test_f1: 0.9804
  test_precision: 0.9691
  test_recall: 0.9921

[비교] Best 단일 모델 (LR) Test AUC : 0.8210
[비교] Soft Voting Ensemble Test AUC: 0.8283


---
## 9. 결과 저장

In [78]:
ensemble_row = pd.DataFrame([{
    '모델'          : 'Soft Voting Ensemble',
    'CV AUC (mean)' : f"{np.mean(ensemble_cv['auc']):.4f}",
    'CV AUC (std)'  : f"{np.std(ensemble_cv['auc']):.4f}",
    'Test AUC'      : f"{ensemble_test['test_auc']:.4f}",
    'Test F1'       : f"{ensemble_test['test_f1']:.4f}",
    'Test Precision': f"{ensemble_test['test_precision']:.4f}",
    'Test Recall'   : f"{ensemble_test['test_recall']:.4f}",
}]).set_index('모델')

comp_df = pd.concat([comp_df, ensemble_row])
comp_df.to_csv('../outputs/model_comparison.csv')
print('[저장] ../outputs/model_comparison.csv')

print('\n=== 최종 모델 비교 (앙상블 포함) ===')
print(comp_df.to_string())

[저장] ../outputs/model_comparison.csv

=== 최종 모델 비교 (앙상블 포함) ===
                     CV AUC (mean) CV AUC (std) Test AUC Test F1 Test Precision Test Recall
모델                                                                                         
Logistic Regression         0.6312       0.0508   0.8210  0.8215         0.9852      0.7045
Decision Tree               0.5071       0.0438   0.4842  0.9683         0.9683      0.9683
Random Forest               0.6715       0.0410   0.8254  0.9844         0.9693      1.0000
LightGBM                    0.6318       0.0705   0.7898  0.9804         0.9691      0.9921
XGBoost                     0.6115       0.0877   0.7874  0.9818         0.9692      0.9947
Soft Voting Ensemble        0.6268       0.0453   0.8283  0.9804         0.9691      0.9921
